# Cross section extraction (v3, adopted extraction)

Used in parallel with `xsec_analyzer`-style reweighting (now `08_systematics.py`) to extract and plot (with systematics)
the $\nu$NCQE cross section. Since it is a single-bin measurement there is no unfolding; the event rates come from the
CV selection of `05_FULL_DATA_signal_band.ipynb` (re-run in `lib/selection.py`, identical yields).

### v3 (2026-10-15): what changed

| | v2 | v3 (paper) |
|---|---|---|
| OOFV $\nu$NCQE (140.4 of the 253.4 OOFV events) | subtracted as background | scaled with $f$ together with the FV signal (T2K, PRD 100, 112009, Eq. 5) |
| beam-uncorrelated (CIT) background | off-beam triggers, 127.44 (no FMV/MRD readout, so no veto) | pre-beam window [0,150) ns of the beam triggers, 67 x 182/150 = 81.29 (as in 03) |
| skyshine | 14.1 +- 13.0 | refitted with the pre-beam CIT: 13.95 +- 12.87 (`08_systematics.py`, stage B) |
| $\langle\sigma_{GENIE}\rangle$ | spline average 2.04334 | MC-implied $N_{true}/(\mathrm{POT}\,\Phi\,N_T)$ = 2.0513 (same simulation as the efficiency) |
| statistical uncertainty | $\sqrt{N}/N$ = 3.6% | $\sqrt{N_{obs}}/(M_{sig}+M^{OOFV}_{\nu NCQE})$ = 6.5% (T2K convention) |
| inputs | typed in by hand from 05 | read from `results/08_systematics.json` and `results/06_primary_gamma.json` |

$$f_{\nu NCQE}=\frac{N_{obs}-M_{bkg}-D_{CIT}-D_{sky}}{M_{sig}+M^{OOFV}_{\nu NCQE}},\qquad
\langle\sigma_{\nu NCQE}\rangle=f_{\nu NCQE}\,\langle\sigma^{GENIE}_{\nu NCQE}\rangle$$

**Run order:** `python3 08_systematics.py` -> `06_Primary_Dexcitation_Systematic.ipynb` -> this notebook.
Systematic terms analysed elsewhere (secondary $\gamma$ rays, PMT tilt, energy scale, vertex, POT) are entered by hand
below and flagged; the rest come from 06 and 08.

In [1]:
from lib.imports import *
import json
from lib import paths, extraction as X

R08 = json.load(open(paths.RESULTS_DIR + '08_systematics.json'))       # inputs, reweighting, bunch timing, backgrounds
R06 = json.load(open(paths.RESULTS_DIR + '06_primary_gamma.json'))     # primary gamma-ray emission (p, s, "others")
for name, R in (('08_systematics.py', R08), ('06_Primary_Dexcitation_Systematic', R06)):
    p = R['provenance']
    print(f"{name:36s} {p['date']}  git {p['git_branch']}@{p['git_commit']}{' (uncommitted changes)' if p['git_dirty'] else ''}")

08_systematics.py                    2026-10-15T07:49:43  git adopted-extraction@6f144c4 (uncommitted changes)
06_Primary_Dexcitation_Systematic    2026-10-15T07:50:30  git adopted-extraction@6f144c4 (uncommitted changes)


In [2]:
# ==========================================
# INPUTS (from 08_systematics.py, stage A/B)
# ==========================================
# Counts are POT-scaled (MC; TunedCV-weighted, MicroBooNE v3.0.6 tune), livetime-scaled (pre-beam) or from the sideband
# fit (skyshine). External (dirt) MC is scaled by alpha_dirt = 0.27 (03).
I = R08['inputs']

# Data parameters
N_DATA_RECO = I['N_obs']                  # observed events, first 26 bunches, +-1 sigma bunch windows

# CIT (pre-beam) + skyshine parameters
N_CIT, N_CIT_ERR = I['D_CIT'], I['D_CIT_err']                   # 67 pre-beam clusters x 182 ns / 150 ns
N_SKYSHINE, N_SKYSHINE_ERR = I['D_sky'], I['D_sky_err']         # skyshine in the signal region (pre-beam CIT refit)

# MC parameters
MC_TRUE_SIGNAL = I['N_true_FV']           # true FV nuNCQE on 16O (POT-scaled): the efficiency denominator
SIGMA_GENIE = I['sigma_genie']            # MC-implied <sigma_GENIE> = N_true / (POT Phi N_T), 1e-38 cm^2/oxygen

# reco (first 26 bunches):
Y = I['yields']
MC_NCQE      = Y['sig']                   # FV nuNCQE (signal)
MC_OOFV_NCQE = Y['OOFV_nu']               # nuNCQE with a true vertex outside the FV: scaled with f (signal-like)
MC_NCother   = Y['NCother']
MC_CC        = Y['CC']
MC_nubar     = Y['nubar']
MC_OOFV      = Y['OOFV_other']            # remaining OOFV interactions
MC_external  = Y['Ext']                   # dirt events scaled by alpha

MC_SIGNAL_LIKE = MC_NCQE + MC_OOFV_NCQE
MC_TOTAL_BKG = MC_NCother + MC_CC + MC_nubar + MC_OOFV + MC_external
MC_TOTAL_RECO = MC_SIGNAL_LIKE + MC_TOTAL_BKG
MC_EFFICIENCY = MC_NCQE / MC_TRUE_SIGNAL   # selection efficiency (FV nuNCQE reco / FV nuNCQE true)

# ==========================================
# CALCULATIONS (paper Eq. 26-27)
# ==========================================
total_bkg = MC_TOTAL_BKG + N_CIT + N_SKYSHINE
scale_factor = (N_DATA_RECO - total_bkg) / MC_SIGNAL_LIKE
sigma_obs = SIGMA_GENIE * scale_factor
STAT_ERROR = 100 * np.sqrt(N_DATA_RECO) / MC_SIGNAL_LIKE / scale_factor   # % (T2K convention)

# base GENIE de-excitation (same extraction, base-model MC), for the model comparison
YB = I['yields_base']
f_base = (N_DATA_RECO - sum(YB[c] for c in ('NCother', 'CC', 'nubar', 'OOFV_other', 'Ext')) - N_CIT - N_SKYSHINE) \
         / (YB['sig'] + YB['OOFV_nu'])
sigma_base = f_base * I['sigma_genie_base']

# ==========================================
# OUTPUT
# ==========================================
print("--- EVENT BREAKDOWN ---")
print(f"Reconstructed events (Data):       {N_DATA_RECO}")
print(f"Total Expected Background:         {total_bkg:.2f}")
print("----------------------------------")
print(f"CIT (pre-beam) background:         {N_CIT:.2f} +/- {N_CIT_ERR:.2f}")
print(f"Fitted skyshine background:        {N_SKYSHINE:.2f} +/- {N_SKYSHINE_ERR:.2f}")
print(f"Predicted MC background:           {MC_TOTAL_BKG:.2f}")
print("----------------------------------")
print(f"Total Predicted MC Signal:         {MC_TRUE_SIGNAL:.2f}")
print(f"Predicted Reco MC Signal (FV):     {MC_NCQE:.2f}")
print(f"Predicted Reco OOFV nuNCQE:        {MC_OOFV_NCQE:.2f}")
print(f"Efficiency:                        {100 * MC_EFFICIENCY:.2f}%")
print("----------------------------------")
print("--- RESULTS ---")
print(f"<sigma_GENIE> (MC-implied):        {SIGMA_GENIE:.4f} e-38  (spline average {X.SIGMA_GENIE_SPLINE})")
print(f"Scale factor (Data/GENIE):         {scale_factor:.4f} +/- {scale_factor * STAT_ERROR / 100:.4f} (stat)")
print(f"Observed XS:                       {sigma_obs:.4f} +/- {sigma_obs * STAT_ERROR / 100:.4f} (stat) e-38 ({STAT_ERROR:.2f}%)")
print(f"Base GENIE de-excitation:          f = {f_base:.4f}, XS = {sigma_base:.4f} e-38 ({100 * (sigma_base / sigma_obs - 1):+.1f}%)")

--- EVENT BREAKDOWN ---
Reconstructed events (Data):       772
Total Expected Background:         344.15
----------------------------------
CIT (pre-beam) background:         81.29 +/- 9.93
Fitted skyshine background:        13.95 +/- 12.87
Predicted MC background:           248.91
----------------------------------
Total Predicted MC Signal:         11324.42
Predicted Reco MC Signal (FV):     152.61
Predicted Reco OOFV nuNCQE:        140.38
Efficiency:                        1.35%
----------------------------------
--- RESULTS ---
<sigma_GENIE> (MC-implied):        2.0513 e-38  (spline average 2.04334)
Scale factor (Data/GENIE):         1.4603 +/- 0.0948 (stat)
Observed XS:                       2.9955 +/- 0.1945 (stat) e-38 (6.49%)
Base GENIE de-excitation:          f = 1.1095, XS = 2.2845 e-38 (-23.7%)


In [3]:
# Investigating systematic impact on the cross section (adjust above script and re-calculate XS)

def XS_systematic(nominal, adjusted):
    print(f"{round(100*np.abs(adjusted - nominal) / nominal,2)}%")

# v2 extraction with the v2 inputs (OOFV nuNCQE as background, off-beam CIT, spline <sigma_GENIE>), for reference only
v2 = X.extract_f_earlier(dict(sig=MC_NCQE, OOFV_nu=MC_OOFV_NCQE, NCother=MC_NCother, CC=MC_CC, nubar=MC_nubar,
                              OOFV_other=MC_OOFV, Ext=MC_external), N_DATA_RECO) * X.SIGMA_GENIE_SPLINE
print(f'v2 extraction (reference): {v2:.4f} e-38;  v3: {sigma_obs:.4f} e-38')
XS_systematic(v2, sigma_obs)

v2 extraction (reference): 3.2293 e-38;  v3: 2.9955 e-38
7.24%


In [4]:
# Plotting

def plot_official_xs(CV_val, CV_label, data_val, BNB_stat_err,
                     total_err_up, total_err_down, 
                     pot_string, label_prefix, plot_name, stat_only,
                     deexc_alt_val=None, deexc_alt_label='Base GENIE de-excitation'):
    """
    ... (existing docstring) ...
    deexc_alt_val (float, optional): Result using an alternate de-excitation
        model in the unfolding (e.g. base GENIE instead of the Empirical Model).
        Shown as a band + open marker, separate from stat/sys error bars.
    deexc_alt_label (str): Legend label for the alt de-excitation marker.
    """
    fig, ax = plt.subplots(figsize=(3.2, 5))
    
    if stat_only:
        print('\nONLY STAT (BNB + MC) ERRORS')
        total_label = f'{label_prefix} (stat. + MC stat.)'
    else:
        total_label = f'{label_prefix} (stat. + sys.)'

    stat_err = (BNB_stat_err/100) * data_val
    tot_err_up   = (total_err_up   / 100) * data_val
    tot_err_down = (total_err_down / 100) * data_val

    # --- De-excitation model dependence (draw first -> lowest zorder) ---
    if deexc_alt_val is not None:
        lo, hi = sorted([data_val, deexc_alt_val])
        ax.plot(0.5, deexc_alt_val, marker='D', markerfacecolor='none',
                markeredgecolor='#FF3366', markeredgewidth=1.2, markersize=6,
                linestyle='none', zorder=4, label = 'Base GENIE de-exc.')

    # --- Inner error bar (Stat only, symmetric) ---
    ax.errorbar(0.5, data_val, yerr=stat_err,
            fmt='ko', capsize=5, label=total_label, markersize=5, zorder=5)

    # --- Outer error bar (Total, asymmetric) ---
    ax.errorbar(0.5, data_val, yerr=[[tot_err_down], [tot_err_up]],
            fmt='none', ecolor='k', capsize=5, zorder=3)

    # --- Prediction (GENIE CV) ---
    ax.axhline(CV_val, label=CV_label, color='blue',
               linestyle='dashdot', linewidth=1.5, zorder=2)

    # --- 3. Formatting Ticks & Scale ---
    ax.tick_params(axis='y', which='both', direction='in', right=True, labelsize=14)
    ax.yaxis.set_major_locator(MultipleLocator(0.5e-38))
    ax.yaxis.set_minor_locator(MultipleLocator(0.1e-38))
    ax.set_xticks([]) # Remove x-axis ticks
    ax.set_xlim(0, 1) # Set arbitrary bounds to center the point at 0.5

    # Force scientific notation 10^-38
    formatter = ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((0,0))
    ax.yaxis.set_major_formatter(formatter)

    # --- 4. Labels & Legend ---
    plt.ylabel(r'$\nu$' + 'NCQE Cross Section [cm' + r'$^2$' + ' / oxygen]', fontsize=14)
    plt.ylim([0, 5.5e-38])
    
    # Legend - frameon=False as requested
    plt.legend(frameon=False, fontsize=9, loc='upper right')

    # --- 5. Bottom Text (POT & Experiment Info) ---
    # Using weight='bold' to match the MicroBooNE style
    bottom_text = f"ANNIE {label_prefix}\n{pot_string} POT"
    ax.text(
        0.05, 0.05, bottom_text,
        transform=ax.transAxes,
        fontsize=11,
        fontweight='bold',
        verticalalignment='bottom'
    )
    
    
    plt.tight_layout()
    plt.savefig(plot_name, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor = 'w')
    plt.show()
    
    print('Plot saved to: plots/' + plot_name)


# ==========================================
# Asymmetric uncertainty combination
# ==========================================
def quad_sys_asymmetric(up_uncs, down_uncs):
    """Combine asymmetric systematics in quadrature separately"""
    return np.sqrt(np.sum(np.array(up_uncs)**2)), \
           np.sqrt(np.sum(np.array(down_uncs)**2))

# Systematics -- (up %, down %) on the cross section; symmetric systematics get the same value both sides
B08 = R08['budget_terms']
ELSEWHERE = '(analysed elsewhere; thesis Ch. 5, evaluated before the OOFV-scaling change -- expected conservative)'
systematics = {
                              # up,   down
    'genie_xsec':      tuple(B08['Interaction model']),        # 08: 44-knob multisim, ratio transfer
    'flux':            tuple(B08['Flux']),                     # 08: 13 flux knobs
    'primary_p_shell': tuple(R06['p_shell']),                  # 06
    'primary_s_shell': tuple(R06['s_shell']),                  # 06
    'primary_others':  (0.0, abs(min(R06['others'], 0.0))),    # 06: T2K-like "other" states, one-sided
    'secondary':       (4.6, 4.6),                             # ELSEWHERE: QGSP_BIC_HP / QGSP_INCL++_HP
    'PMT_tilt':        (3.1, 3.1),                             # ELSEWHERE: untilted geometry
    'energy_scale':    (7.0, 7.0),                             # ELSEWHERE: r_CE^norm = 0.95
    'vertex':          (3.0, 3.0),                             # ELSEWHERE: 10 cm vertex smearing
    'bunch_timing':    tuple(B08['Bunch timing']),             # 08: common + relative shift (+) width / narrowing
    'dirt':            tuple(B08['External normalization']),   # 08: alpha_dirt +- |alpha_joint - alpha_CCinc|
    'skyshine':        tuple(B08['Skyshine']),                 # 08
    'CIT_stat':        tuple(B08['Pre-beam (CIT) statistics']),# 08
    'MC_stat':         tuple(B08['MC statistics']),            # 08: delta method on the CV sample
    'POT':             (2.0, 2.0),                             # ELSEWHERE: TOR860 / TOR875 agreement
}
GROUPS = {'primary gamma': ['primary_p_shell', 'primary_s_shell', 'primary_others'],
          'detector response': ['PMT_tilt', 'energy_scale', 'vertex', 'bunch_timing']}

up_sys   = [v[0] for v in systematics.values()]
down_sys = [v[1] for v in systematics.values()]
sys_up, sys_down = quad_sys_asymmetric(up_sys, down_sys)

#STAT_ERROR = 0     # enable for syst-only
TOTAL_UP   = np.sqrt(STAT_ERROR**2 + sys_up**2)
TOTAL_DOWN = np.sqrt(STAT_ERROR**2 + sys_down**2)

print(f"{'term':18s} {'up %':>7s} {'down %':>7s}")
for k, (u, d) in systematics.items():
    print(f"{k:18s} {u:7.2f} {d:7.2f}" + ('   ' + ELSEWHERE if k in ('secondary', 'PMT_tilt', 'energy_scale', 'vertex') else ''))
for g, ks in GROUPS.items():
    gu, gd = quad_sys_asymmetric([systematics[k][0] for k in ks], [systematics[k][1] for k in ks])
    print(f"  -> {g:15s} +{gu:.2f} / -{gd:.2f}")
print(f"Stat error:       ±{STAT_ERROR:.2f}%")
print(f"Sys up:           +{sys_up:.2f}%")
print(f"Sys down:         -{sys_down:.2f}%")
print(f"Total upper:      +{TOTAL_UP:.2f}%")
print(f"Total lower:      -{TOTAL_DOWN:.2f}%")
print(f"\nf     = {scale_factor:.2f} ± {scale_factor * STAT_ERROR / 100:.2f} (stat) +{scale_factor * sys_up / 100:.2f} "
      f"-{scale_factor * sys_down / 100:.2f} (syst)")
print(f"sigma = {sigma_obs:.2f} ± {sigma_obs * STAT_ERROR / 100:.2f} (stat) +{sigma_obs * sys_up / 100:.2f} "
      f"-{sigma_obs * sys_down / 100:.2f} (syst) x 1e-38 cm^2/oxygen")

with open(paths.RESULTS_DIR + '07_budget.json', 'w') as fh:
    json.dump(dict(provenance=X.provenance(dict(notebook='07_NCQE_XS_Extraction_Plotting.ipynb')),
                   f=scale_factor, sigma=sigma_obs, sigma_genie=SIGMA_GENIE, sigma_base=sigma_base, stat=STAT_ERROR,
                   systematics=systematics, syst=[sys_up, sys_down], total=[TOTAL_UP, TOTAL_DOWN]), fh, indent=1, default=float)

os.makedirs('plots', exist_ok=True)
plot_official_xs(
    CV_val=SIGMA_GENIE * 1e-38,
    CV_label='GENIE v3.0.6',
    data_val=sigma_obs * 1e-38,
    BNB_stat_err=STAT_ERROR,
    total_err_up=TOTAL_UP,
    total_err_down=TOTAL_DOWN,
    pot_string=r"$\mathbf{" + f"{I['POT_1e20']:.2f}" + r"\times10^{20}}$",
    label_prefix="Data",
    plot_name="plots/NCQE_XS.png",
    stat_only=False,
    deexc_alt_val=sigma_base * 1e-38,     # base GENIE de-excitation, same extraction
    deexc_alt_label="Base GENIE de-exc."
)

term                  up %  down %
genie_xsec            8.60    8.60
flux                  9.95    9.95
primary_p_shell      10.90    3.16
primary_s_shell       1.61    1.63
primary_others        0.00   15.23
secondary             4.60    4.60   (analysed elsewhere; thesis Ch. 5, evaluated before the OOFV-scaling change -- expected conservative)
PMT_tilt              3.10    3.10   (analysed elsewhere; thesis Ch. 5, evaluated before the OOFV-scaling change -- expected conservative)
energy_scale          7.00    7.00   (analysed elsewhere; thesis Ch. 5, evaluated before the OOFV-scaling change -- expected conservative)
vertex                3.00    3.00   (analysed elsewhere; thesis Ch. 5, evaluated before the OOFV-scaling change -- expected conservative)
bunch_timing         12.32    7.39
dirt                  5.23    5.23
skyshine              3.01    3.01
CIT_stat              2.32    2.32
MC_stat               5.18    5.18
POT                   2.00    2.00
  -> primary gamma   +11

<Figure size 320x500 with 1 Axes>

Plot saved to: plots/plots/NCQE_XS.png
